# 08. 확실한 어절 고정 + 앞뒤 문맥·유사도로 나머지 추론 (공개 언어모델)
아이디어: 이전 최고 모델 결과에서 확실한 단어를 먼저 고정하고, 나머지를 정방향·역방향 문맥과 유사도를 함께 보며 하나씩 추론.

1. **고정**: 어절 단위로 01 결과와 v2 태거 결과가 같으면 고정 (검증 기준 일치 91.4%, 정확도 98.2%)
   - 원안(글자 수 기준)도 `--anchor length --min_len 4` 로 비교 (로컬 50문장 확인: 4글자 이상 고정 시 고정 어절 정확도 90.3% 로 낮음)
2. **후보**: 불일치 어절마다 {01 결과, v2 결과, 01 상위 후보 6개}
3. **점수** = 유사도(01 채널: 이 후보가 입력 형태로 난독화될 확률) + w × 문맥(공개 LM 이 본 앞뒤 120자 창 전체의 자연스러움)
4. **순서**: 긴 어절부터 확정 → 확정 결과를 다음 어절의 문맥으로 사용. 2회차는 역방향으로 재결정

| 비교 기준 | 검증 F1 |
|---|---|
| 01 | 0.9834 |
| 07 재순위화 (1.5B) | 0.9843 |

**전제:** 05 의 v2 태거(처음부터) 검증 예측이 있어야 함 (`val_pred_tagger_v2_scratch.csv`). test 에는 그 test 예측도 필요 → 아래 0-1 셀.
**GPU 런타임(L4 권장)**, 첫 셀부터.

## 0) 준비

In [ ]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' --exclude 'v2_cache_*.pkl' "$DRIVE/" /content/work/
%cd /content/work
!ls outputs | grep -E 'val_pred.csv|val_pred_tagger_v2_scratch|submission_tagger_v2_scratch|submission_noisy_channel.csv|tagger_v2_best_scratch'; nproc
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    # --update: 드라이브 쪽이 더 최신이면 덮어쓰지 않음 (다른 노트북과 동시에 돌려도 안전)
    !rsync -a --update --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    print('saved outputs -> drive')

### 0-1) (test 용, 한 번만) v2 태거의 test 예측 만들기
위 목록에 `submission_tagger_v2_scratch.csv` 가 없을 때만 실행.

In [ ]:
!python -u tagger_v2.py --mode predict --run_name scratch --workers 8
save_back()

## 1) 빠른 확인 (검증 100문장)

In [ ]:
!python -u anchor_infill.py --mode val --val_limit 100

## 2) 검증 전체: 일치 기준 → `best_anchor_<설정>.json`
출력에서 확인: 고정 어절 정확도, '추론 대상 중 01 이 맞은 비율', 후보 상한, w 별 F1.

In [ ]:
!python -u anchor_infill.py --mode val
save_back()

In [ ]:
!python -u anchor_infill.py --mode val --weights 0.75,0.8,0.9,1.0,1.25,1.5,1.75,2,2.5,3,5
save_back()


In [ ]:
!python -u anchor_infill.py --mode val --weights 1.5 --freq_weights 0,0.25,0.5,1,2 --margins 0,1,2,4 --name 09
save_back()


In [ ]:
!python -u anchor_infill.py --mode test --name 09
save_back()

### (비교) 사용자 원안: 4글자 이상 고정

In [ ]:
!python -u anchor_infill.py --mode val --anchor length --min_len 4
save_back()

In [ ]:
import pandas as pd
r = pd.read_csv('outputs/anchor_runs.csv')
r[r.n_val == r.n_val.max()].sort_values('f1', ascending=False)[
    ['anchor', 'name', 'w', 'w_freq', 'margin', 'f1', 'base_f1', 'oracle_f1', 'sec']].head(20)


## 3) 01 / 07 / 08 비교

In [ ]:
import glob
a = pd.read_csv('outputs/val_pred.csv').set_index('ID')
print(f"01                         F1 {a.f1.mean():.4f}")
for f in ['outputs/val_pred_rerank_Qwen2.5-1.5B.csv'] + sorted(glob.glob('outputs/val_pred_anchor_*.csv')):
    b = pd.read_csv(f).set_index('ID'); b['pred'] = b['pred'].fillna('')
    ids = a.index.intersection(b.index); d = b.loc[ids, 'f1'] - a.loc[ids, 'f1']
    print(f"{f.split('val_pred_')[1][:-4]:26s} F1 {b.loc[ids].f1.mean():.4f} (rows {len(ids)}) | 좋아진 행 {(d>0).sum()} / 나빠진 행 {(d<0).sum()}")
b = pd.read_csv(sorted(glob.glob('outputs/val_pred_anchor_agree*.csv'))[0]).set_index('ID')
d = b.f1 - a.f1
for i in d.sort_values().index[:2].tolist() + d.sort_values().index[-3:].tolist():
    print(f'[{d[i]:+.3f}] GT :', b.loc[i, 'output'][:90]); print('        01 :', a.loc[i, 'pred'][:90]); print('        08 :', b.loc[i, 'pred'][:90])

## 4) test 예측 → `outputs/submission_anchor_<설정>.csv`
(0-1 의 v2 test 예측 필요)

In [ ]:
!python -u anchor_infill.py --mode test
save_back()

### (선택) 런타임 해제

In [ ]:
from google.colab import runtime; runtime.unassign()